# 02 · Train turn-detection experiment (E1-E6)

**Kaggle settings:** GPU **T4 x2 or P100** · **Internet ON** (downloads
whisper-tiny once) · ~1-2 h per experiment.

**Attach as input datasets** (Add Input):
1. `smart-turn-enhi-prep` — output of notebook 01
2. `hinglish-synth` — the uploaded synthetic Hinglish dataset
3. *(only when resuming)* the previous version's output of THIS notebook

**Run an experiment:** set `EXPERIMENT` in the config cell to one of
`e1_baseline` · `e2_hinglish_aug` · `e3_tinymel_scratch` · `e4_no_pause_aug` ·
`e5_distill` · `e6_full_data`, then *Save Version → Save & Run All*. Repeat per
experiment (one per session).

**Distillation (E5):** `e5_distill` trains TinyMelNet against a frozen Whisper
teacher, so it additionally needs the teacher's `ckpt_best.pt`. Attach the
`turn-detect-ckpt` dataset and set `TEACHER_FROM` to the teacher's run folder
(e.g. `/kaggle/input/turn-detect-ckpt/run_e2_hinglish_aug`);
`python -m tools.push_kaggle train e5_distill --teacher e2_hinglish_aug` stages
that checkpoint and writes the path for you. The run raises immediately if the
teacher is missing rather than silently training without it.

**Time budget:** `TIME_BUDGET_MIN` (default 630 = 10.5 h) makes training stop
itself at the next checkpoint before Kaggle's 12 h session wall. This matters:
a commit run killed by the wall publishes **no output at all**, so a run that
would overrun must end early and leave `ckpt_last.pt` behind to resume from.

**Resume after a kill (or a time-budget stop):** attach the previous run's
output (or the `turn-detect-ckpt` dataset built by
`python -m tools.push_kaggle train <exp> --resume`), set `RESUME_FROM` to its
path (e.g. `/kaggle/input/turn-detect-ckpt/run_e2_hinglish_aug`), run again.
Training continues from the last checkpoint (≤500 steps lost). A missing
checkpoint or a config-hash mismatch now raises instead of silently
restarting from scratch.

**Afterwards:** download `run_<EXPERIMENT>/` (metrics.json, ckpt_best.pt,
model_fp32.onnx, model_int8.onnx) into the repo's `experiments/` folder.

The `turn_detector` package below is auto-generated from the tested repo
sources by `tools/build_notebooks.py` — edit the repo, not the cells.


In [ ]:
%pip install -q onnx onnxruntime onnxscript polars soundfile

In [ ]:
import os
os.makedirs("turn_detector", exist_ok=True)

In [ ]:
%%writefile turn_detector/__init__.py
"""Tiny audio turn detection (Hinglish-focused)."""

In [ ]:
%%writefile turn_detector/common.py
"""Torch-free audio constants + windowing shared by training and inference."""

import numpy as np

SAMPLE_RATE = 16000
WINDOW_SECONDS = 8.0
N_SAMPLES = int(SAMPLE_RATE * WINDOW_SECONDS)   # 128000
N_FFT = 400
HOP = 160
N_MELS = 80
N_FRAMES = N_SAMPLES // HOP                      # 800 mel frames
N_ENCODER_POSITIONS = N_FRAMES // 2              # 400 after Whisper's stride-2 conv


def right_align(wav: np.ndarray, n_samples: int = N_SAMPLES) -> np.ndarray:
    """Keep the last n_samples; left-pad with zeros if shorter."""
    wav = wav[-n_samples:]
    if len(wav) < n_samples:
        wav = np.concatenate([np.zeros(n_samples - len(wav), dtype=wav.dtype), wav])
    return wav.astype(np.float32)

In [ ]:
%%writefile turn_detector/features.py
"""Whisper-compatible log-mel frontend, windowed to the LAST 8 seconds.

Reimplements transformers' WhisperFeatureExtractor math in torch (batchable,
GPU-capable) instead of numpy: STFT(n_fft=400, hop=160, hann, center/reflect),
drop last frame, slaney mel (80 bins, 0-8kHz), log10 -> clamp to max-8 ->
(x+4)/4. Parity with the HF extractor is asserted in tests/test_features.py.

Turn detection cares about how speech ENDS, so windows are right-aligned:
the final audio sample always lands at the last mel frame; short clips are
zero-padded on the LEFT.
"""

import torch
import torch.nn as nn

from turn_detector.common import (  # noqa: F401  (re-exported for callers)
    HOP, N_ENCODER_POSITIONS, N_FFT, N_FRAMES, N_MELS, N_SAMPLES,
    SAMPLE_RATE, WINDOW_SECONDS, right_align,
)


class LogMel(nn.Module):
    """waveform (B, 128000) float32 -> log-mel (B, 80, 800)."""

    def __init__(self):
        super().__init__()
        from transformers.audio_utils import mel_filter_bank
        filters = mel_filter_bank(
            num_frequency_bins=1 + N_FFT // 2,
            num_mel_filters=N_MELS,
            min_frequency=0.0,
            max_frequency=8000.0,
            sampling_rate=SAMPLE_RATE,
            norm="slaney",
            mel_scale="slaney",
        )  # (201, 80)
        self.register_buffer("mel_filters", torch.from_numpy(filters).float())
        self.register_buffer("window", torch.hann_window(N_FFT, periodic=True))

    @torch.no_grad()
    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        if wav.dim() == 1:
            wav = wav.unsqueeze(0)
        stft = torch.stft(
            wav, N_FFT, HOP, window=self.window,
            center=True, pad_mode="reflect", return_complex=True,
        )
        magnitudes = stft[..., :-1].abs() ** 2                  # (B, 201, 800)
        mel = self.mel_filters.T @ magnitudes                   # (B, 80, 800)
        log_spec = torch.clamp(mel, min=1e-10).log10()
        log_spec = torch.maximum(
            log_spec, log_spec.amax(dim=(1, 2), keepdim=True) - 8.0
        )
        return (log_spec + 4.0) / 4.0

In [ ]:
%%writefile turn_detector/augment.py
"""Waveform-level augmentations for turn detection.

The two augmentations that teach "silence != done":
  - trailing_silence: appended to ANY clip without changing its label — a
    finished speaker followed by silence is still finished, an unfinished one
    is still unfinished.
  - pause_cut: truncate a COMPLETE utterance at a speech-active point and
    append silence -> a genuine "paused mid-thought" example (label flips to
    incomplete). Applied on the fly in the dataset.

All functions take/return float32 mono @16kHz and are deterministic given rng.
"""

import numpy as np

SR = 16000


def trailing_silence(wav: np.ndarray, rng: np.random.Generator,
                     min_s: float = 0.2, max_s: float = 1.2) -> np.ndarray:
    n = int(rng.uniform(min_s, max_s) * SR)
    return np.concatenate([wav, np.zeros(n, dtype=np.float32)])


def _energy_envelope(wav: np.ndarray, frame: int = 400, hop: int = 160):
    n_frames = max(1, (len(wav) - frame) // hop + 1)
    idx = np.arange(n_frames)[:, None] * hop + np.arange(frame)[None, :]
    return np.sqrt((wav[idx.clip(max=len(wav) - 1)] ** 2).mean(axis=1)), hop


def pause_cut(wav: np.ndarray, rng: np.random.Generator,
              lo: float = 0.4, hi: float = 0.85,
              min_keep_s: float = 0.6, min_removed_s: float = 0.4):
    """Cut a complete utterance mid-speech. Returns wav or None if impossible."""
    env, hop = _energy_envelope(wav)
    thresh = max(env.max() * 0.15, 1e-4)
    active = np.nonzero(env > thresh)[0]
    if len(active) < 10:
        return None
    span_start, span_end = active[0], active[-1]
    span = span_end - span_start
    if span < 20:
        return None
    # candidate frames inside [lo, hi] of the active span that are speech-active
    frame_lo = span_start + int(span * lo)
    frame_hi = span_start + int(span * hi)
    candidates = active[(active >= frame_lo) & (active <= frame_hi)]
    if len(candidates) == 0:
        return None
    cut_frame = int(rng.choice(candidates))
    cut = cut_frame * hop
    if cut < min_keep_s * SR or len(wav) - cut < min_removed_s * SR:
        return None
    out = wav[:cut]
    return trailing_silence(out, rng)


def add_noise(wav: np.ndarray, rng: np.random.Generator,
              snr_lo: float = 10.0, snr_hi: float = 30.0) -> np.ndarray:
    rms = np.sqrt((wav ** 2).mean())
    if rms < 1e-5:
        return wav
    snr = rng.uniform(snr_lo, snr_hi)
    noise_rms = rms / (10 ** (snr / 20))
    return (wav + rng.normal(0, noise_rms, len(wav))).astype(np.float32)


def speed_perturb(wav: np.ndarray, rng: np.random.Generator,
                  lo: float = 0.9, hi: float = 1.1) -> np.ndarray:
    factor = rng.uniform(lo, hi)
    n_out = int(len(wav) / factor)
    x_old = np.arange(len(wav))
    x_new = np.linspace(0, len(wav) - 1, n_out)
    return np.interp(x_new, x_old, wav).astype(np.float32)

In [ ]:
%%writefile turn_detector/config.py
"""Experiment configurations E1-E6."""

import hashlib
import json
from dataclasses import asdict, dataclass, field


@dataclass
class ExperimentConfig:
    name: str
    arch: str = "whisper"              # whisper | tinymel
    use_hinglish_synth: bool = False

    # augmentation probabilities (train only)
    pause_cut_p: float = 0.0           # complete -> cut mid-speech, label flips to 0
    trailing_silence_p: float = 0.0
    noise_p: float = 0.0
    speed_p: float = 0.0

    # knowledge distillation (empty kd_teacher = plain supervised training).
    # These stay IN config_hash: they define the experiment, and a checkpoint
    # trained against a different teacher/temperature is not resumable as this
    # one. Switching kd_teacher therefore invalidates the old checkpoint, which
    # is the correct behaviour.
    kd_teacher: str = ""               # experiment name whose ckpt_best is the teacher
    kd_alpha: float = 0.3              # weight on hard-label BCE; 1-alpha on the soft term
    kd_temperature: float = 2.0

    # optimization
    epochs: int = 4
    batch_size: int = 64
    lr_encoder: float = 1e-5
    lr_head: float = 1e-4
    weight_decay: float = 0.01
    warmup_frac: float = 0.05
    grad_clip: float = 1.0
    seed: int = 42

    # bookkeeping
    checkpoint_every_steps: int = 500
    notes: str = ""

    def config_hash(self) -> str:
        """Hash of the fields that change training maths.

        Cosmetic/operational fields (`notes`, `checkpoint_every_steps`) are
        excluded so editing a comment or checkpoint cadence does not invalidate
        an in-flight run's resumable checkpoint.
        """
        d = asdict(self)
        d.pop("notes", None)
        d.pop("checkpoint_every_steps", None)
        return hashlib.sha1(
            json.dumps(d, sort_keys=True).encode()
        ).hexdigest()[:10]


EXPERIMENTS = {
    "e1_baseline": ExperimentConfig(
        name="e1_baseline",
        notes="EN+HI real data only, no augmentation, WhisperTinyTurn",
    ),
    "e2_hinglish_aug": ExperimentConfig(
        name="e2_hinglish_aug",
        use_hinglish_synth=True,
        pause_cut_p=0.15,
        trailing_silence_p=0.5,
        noise_p=0.25,
        speed_p=0.25,
        notes="headline model: +hinglish synth, +pause/silence/noise/speed aug",
    ),
    "e3_tinymel_scratch": ExperimentConfig(
        name="e3_tinymel_scratch",
        arch="tinymel",
        use_hinglish_synth=True,
        pause_cut_p=0.15,
        trailing_silence_p=0.5,
        noise_p=0.25,
        speed_p=0.25,
        epochs=8,
        lr_head=3e-4,
        notes="from-scratch ~1M param model, same data/aug as e2",
    ),
    "e4_no_pause_aug": ExperimentConfig(
        name="e4_no_pause_aug",
        use_hinglish_synth=True,
        pause_cut_p=0.0,
        trailing_silence_p=0.0,
        noise_p=0.25,
        speed_p=0.25,
        notes="ablation: e2 minus pause_cut and trailing_silence",
    ),
    "e5_distill": ExperimentConfig(
        name="e5_distill",
        arch="tinymel",
        use_hinglish_synth=True,
        pause_cut_p=0.15,
        trailing_silence_p=0.5,
        noise_p=0.25,
        speed_p=0.25,
        epochs=8,
        lr_head=3e-4,
        kd_teacher="e2_hinglish_aug",
        notes="e3 recipe + distillation from the whisper teacher's soft targets",
    ),
    "e6_full_data": ExperimentConfig(
        name="e6_full_data",
        use_hinglish_synth=True,
        pause_cut_p=0.15,
        trailing_silence_p=0.5,
        noise_p=0.25,
        speed_p=0.25,
        notes="E2 recipe on expanded multilingual prep",
    ),
}

In [ ]:
%%writefile turn_detector/model.py
"""Turn-detection model architectures.

- WhisperTinyTurn: pretrained Whisper-Tiny encoder truncated to an 8s input
  window (positional embeddings sliced 1500 -> 400), + attention pooling +
  small MLP head. ~8M params.
- TinyMelNet: from-scratch comparison, ~1M params: depthwise-separable Conv1d
  stack over mel frames + BiGRU + the same pooling/head.

Both take log-mel (B, 80, 800) and return a single logit per example
(sigmoid -> P(turn complete)).
"""

import torch
import torch.nn as nn

from turn_detector.features import N_ENCODER_POSITIONS


class AttnPool(nn.Module):
    """Learned-query attention pooling over time: (B, T, D) -> (B, D)."""

    def __init__(self, dim: int):
        super().__init__()
        self.query = nn.Parameter(torch.randn(dim) * 0.02)
        self.scale = dim ** -0.5

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(x @ self.query * self.scale, dim=1)
        return (weights.unsqueeze(-1) * x).sum(dim=1)


def make_head(dim: int, hidden: int = 256, dropout: float = 0.1) -> nn.Module:
    return nn.Sequential(
        nn.LayerNorm(dim),
        nn.Linear(dim, hidden),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(hidden, 1),
    )


def truncate_whisper_encoder(encoder, n_positions: int = N_ENCODER_POSITIONS):
    """Slice Whisper's 30s positional table to our window length, in place."""
    old = encoder.embed_positions
    new = nn.Embedding(n_positions, old.embedding_dim)
    new.weight.data.copy_(old.weight.data[:n_positions])
    encoder.embed_positions = new
    encoder.config.max_source_positions = n_positions
    if hasattr(encoder, "max_source_positions"):
        encoder.max_source_positions = n_positions
    return encoder


class WhisperTinyTurn(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        dim = encoder.config.d_model
        self.pool = AttnPool(dim)
        self.head = make_head(dim)

    @classmethod
    def from_pretrained(cls, name: str = "openai/whisper-tiny"):
        from transformers import WhisperModel
        encoder = WhisperModel.from_pretrained(name).encoder
        return cls(truncate_whisper_encoder(encoder))

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        hidden = self.encoder(mel).last_hidden_state       # (B, 400, 384)
        return self.head(self.pool(hidden)).squeeze(-1)


class DSConvBlock(nn.Module):
    """Depthwise-separable Conv1d + BN + GELU."""

    def __init__(self, channels: int, kernel: int = 5, stride: int = 1):
        super().__init__()
        self.depthwise = nn.Conv1d(
            channels, channels, kernel, stride=stride,
            padding=kernel // 2, groups=channels,
        )
        self.pointwise = nn.Conv1d(channels, channels, 1)
        self.norm = nn.BatchNorm1d(channels)
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.norm(self.pointwise(self.depthwise(x))))


class TinyMelNet(nn.Module):
    def __init__(self, width: int = 192, gru_hidden: int = 128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(80, width, 5, stride=2, padding=2),
            nn.BatchNorm1d(width),
            nn.GELU(),
            DSConvBlock(width, stride=2),
            DSConvBlock(width, stride=2),
            DSConvBlock(width, stride=1),
        )                                                   # (B, width, 100)
        self.gru = nn.GRU(width, gru_hidden, batch_first=True, bidirectional=True)
        self.pool = AttnPool(2 * gru_hidden)
        self.head = make_head(2 * gru_hidden)

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        x = self.stem(mel).transpose(1, 2)                  # (B, 100, width)
        x, _ = self.gru(x)                                  # (B, 100, 2*hidden)
        return self.head(self.pool(x)).squeeze(-1)


def build_model(arch: str) -> nn.Module:
    if arch == "whisper":
        return WhisperTinyTurn.from_pretrained()
    if arch == "tinymel":
        return TinyMelNet()
    raise ValueError(f"unknown arch: {arch}")


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [ ]:
%%writefile turn_detector/dataset.py
"""Dataset over FLAC shards + parquet manifests.

Works identically on Kaggle (prep-notebook output + hinglish-synth dataset)
and locally (small subsets, unit tests). Multiple sources are concatenated;
each manifest row needs: id, path, label, language, split — optional:
midfiller, endfiller, synthetic, kind, source.

Augmentation policy (train split only) comes from ExperimentConfig. pause_cut
flips a complete example's label to incomplete on the fly; the sampler
compensates so the effective batch balance stays ~50/50.
"""

from pathlib import Path

import numpy as np
import polars as pl
import soundfile as sf
import torch
from torch.utils.data import Dataset, WeightedRandomSampler

from turn_detector import augment
from turn_detector.config import ExperimentConfig
from turn_detector.features import N_SAMPLES, right_align

OPTIONAL_COLS = {
    "midfiller": False, "endfiller": False, "synthetic": False,
    "kind": "", "source": "", "language": "",
}


def load_manifests(sources: list[tuple[str, str]], split: str) -> pl.DataFrame:
    """sources: [(manifest_parquet_path, audio_root), ...] -> unified frame."""
    frames = []
    for manifest_path, audio_root in sources:
        df = pl.read_parquet(manifest_path).filter(pl.col("split") == split)
        for col, default in OPTIONAL_COLS.items():
            if col not in df.columns:
                df = df.with_columns(pl.lit(default).alias(col))
            else:
                df = df.with_columns(pl.col(col).fill_null(default))
        df = df.with_columns(pl.lit(str(audio_root)).alias("audio_root"))
        frames.append(df.select(
            "id", "path", "label", "language", "split",
            "midfiller", "endfiller", "synthetic", "kind", "source", "audio_root",
        ))
    return pl.concat(frames)


class TurnDataset(Dataset):
    def __init__(self, manifest: pl.DataFrame, cfg: ExperimentConfig,
                 train: bool, seed_offset: int = 0):
        self.rows = manifest.to_dicts()
        self.cfg = cfg
        self.train = train
        self.seed_offset = seed_offset
        self.epoch = 0

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i: int):
        row = self.rows[i]
        wav, sr = sf.read(Path(row["audio_root"]) / row["path"], dtype="float32")
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        label = int(row["label"])

        if self.train:
            rng = np.random.default_rng(
                (self.cfg.seed + self.seed_offset) * 1_000_003
                + self.epoch * 101 + i
            )
            c = self.cfg
            if label == 1 and rng.random() < c.pause_cut_p:
                cut = augment.pause_cut(wav, rng)
                if cut is not None:
                    wav, label = cut, 0
            if rng.random() < c.trailing_silence_p:
                wav = augment.trailing_silence(wav, rng)
            if rng.random() < c.speed_p:
                wav = augment.speed_perturb(wav, rng)
            if rng.random() < c.noise_p:
                wav = augment.add_noise(wav, rng)

        wav = right_align(wav, N_SAMPLES)
        return torch.from_numpy(wav), torch.tensor(label, dtype=torch.float32), i

    def balanced_sampler(self, num_samples: int | None = None) -> WeightedRandomSampler:
        """50/50 sampler; completes oversampled to offset pause_cut label flips."""
        labels = np.array([r["label"] for r in self.rows])
        n_pos, n_neg = int(labels.sum()), int((1 - labels).sum())
        # fraction of drawn completes that stay complete after pause_cut
        keep = 1.0 - self.cfg.pause_cut_p if self.train else 1.0
        target_pos_draw = 0.5 / keep if keep > 0 else 0.5
        target_pos_draw = min(target_pos_draw, 0.75)
        w_pos = target_pos_draw / max(n_pos, 1)
        w_neg = (1 - target_pos_draw) / max(n_neg, 1)
        weights = np.where(labels == 1, w_pos, w_neg)
        return WeightedRandomSampler(
            torch.from_numpy(weights).double(),
            num_samples=num_samples or len(self.rows),
            replacement=True,
        )

In [ ]:
%%writefile turn_detector/train.py
"""Training, evaluation, and ONNX export for turn-detection experiments.

Step-based training (sampler draws with replacement, so batches are iid and a
mid-run resume just continues from the saved step — no epoch bookkeeping).
Checkpoints every cfg.checkpoint_every_steps to <out_dir>/ckpt_last.pt; a
killed Kaggle session loses at most that many steps. Resume is automatic when
the checkpoint's config hash matches. Pass `time_budget_minutes` to have the
run end itself at the next checkpoint before Kaggle's session wall — a commit
run that is killed by the wall publishes no output at all.

Final artifacts in out_dir: ckpt_best.pt, model_fp32.onnx, model_int8.onnx,
metrics.json.

When cfg.kd_teacher is set the loss is blended with a frozen teacher's soft
targets (E5); `teacher_dir` must then point at that experiment's ckpt_best.pt.

Note for security scanners: `model.eval()` below is PyTorch's inference-mode
switch, not code evaluation.
"""

import json
import math
import os
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
from torch.utils.data import DataLoader

from turn_detector.config import ExperimentConfig
from turn_detector.dataset import TurnDataset, load_manifests
from turn_detector.features import LogMel
from turn_detector.model import build_model, count_params


# ---------------- metrics ----------------

def _midranks(values: np.ndarray) -> np.ndarray:
    """1-based ranks with ties averaged (the midrank convention).

    Plain double-argsort breaks ties arbitrarily, which biases AUC whenever
    scores collide (common after int8 quantisation or a saturated sigmoid).
    """
    order = np.argsort(values, kind="mergesort")
    sorted_vals = values[order]
    # first[i] = index of the first element equal to sorted_vals[i]
    _, first_idx, counts = np.unique(sorted_vals, return_index=True,
                                     return_counts=True)
    group = np.repeat(np.arange(len(counts)), counts)
    # mean of the 1-based positions spanned by each tie group
    starts = first_idx[group] + 1
    sizes = counts[group]
    sorted_ranks = starts + (sizes - 1) / 2.0
    ranks = np.empty(len(values), dtype=float)
    ranks[order] = sorted_ranks
    return ranks


def rank_auc(labels: np.ndarray, scores: np.ndarray) -> float:
    """ROC-AUC via the rank-sum statistic (no sklearn dependency)."""
    labels = np.asarray(labels)
    scores = np.asarray(scores, dtype=float)
    pos = scores[labels == 1]
    neg = scores[labels == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    ranks = _midranks(np.concatenate([pos, neg]))
    return float((ranks[: len(pos)].sum() - len(pos) * (len(pos) + 1) / 2)
                 / (len(pos) * len(neg)))


def f1_score(labels: np.ndarray, preds: np.ndarray) -> float:
    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    denom = 2 * tp + fp + fn
    return float(2 * tp / denom) if denom else float("nan")


def tune_threshold(labels: np.ndarray, probs: np.ndarray) -> float:
    best_t, best_acc = 0.5, 0.0
    for t in np.arange(0.05, 0.96, 0.01):
        acc = float(((probs >= t) == labels).mean())
        if acc > best_acc:
            best_acc, best_t = acc, float(t)
    return best_t


def slice_metrics(rows: pl.DataFrame, labels: np.ndarray, probs: np.ndarray,
                  threshold: float) -> dict:
    def compute(mask: np.ndarray) -> dict:
        if mask.sum() == 0:
            return {"n": 0}
        l, p = labels[mask], probs[mask]
        preds = (p >= threshold).astype(int)
        return {
            "n": int(mask.sum()),
            "acc_050": round(float(((p >= 0.5) == l).mean()), 4),
            "acc_tuned": round(float((preds == l).mean()), 4),
            "f1_tuned": round(f1_score(l, preds), 4),
            "auc": round(rank_auc(l, p), 4),
        }

    lang = rows["language"].to_numpy()
    midf = rows["midfiller"].fill_null(False).to_numpy().astype(bool)
    endf = rows["endfiller"].fill_null(False).to_numpy().astype(bool)
    synth = rows["synthetic"].fill_null(False).to_numpy().astype(bool)
    all_mask = np.ones(len(labels), dtype=bool)
    return {
        "overall": compute(all_mask),
        "english": compute(lang == "english"),
        "hindi": compute(lang == "hindi"),
        "hinglish": compute(lang == "hinglish"),
        # E6's prep adds a multilingual training tail whose manifest language is
        # the raw ISO code; the v3.2 test split is EN+HI only, so this slice is
        # n=0 there by construction — reported rather than hidden.
        "multilingual_other": compute(
            ~np.isin(lang, ("english", "hindi", "hinglish"))),
        "filler": compute(midf | endf),
        "human_audio": compute(~synth & (lang != "hinglish")),
        "threshold": threshold,
    }


# ---------------- eval ----------------

@torch.no_grad()
def predict(model, mel_fn, dataset, device, batch_size=64, num_workers=2):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=(device != "cpu"))
    model.eval()
    all_probs, all_labels = [], []
    for wav, label, _ in loader:
        mel = mel_fn(wav.to(device))
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            logits = model(mel)
        all_probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        all_labels.append(label.numpy())
    return np.concatenate(all_labels), np.concatenate(all_probs)


# ---------------- checkpointing ----------------

def atomic_save(obj, path: Path):
    """Write via a sibling tmp file + os.replace.

    A Kaggle session killed mid-`torch.save` would otherwise leave a truncated
    ckpt_last.pt and make the next run unresumable.
    """
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def save_ckpt(path: Path, model, opt, sched, scaler, step, best_val_auc, cfg_hash):
    atomic_save({
        "model": model.state_dict(), "opt": opt.state_dict(),
        "sched": sched.state_dict(), "scaler": scaler.state_dict(),
        "step": step, "best_val_auc": best_val_auc, "cfg_hash": cfg_hash,
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }, path)


# ---------------- distillation ----------------

def load_teacher(cfg: ExperimentConfig, teacher_dir: str | None, device: str):
    """Frozen teacher for distillation: <teacher_dir>/ckpt_best.pt.

    Required — never optional — whenever cfg.kd_teacher is set: silently
    training a "distilled" student against no teacher would burn a GPU session
    and produce a run whose name lies about what it is.
    """
    if not teacher_dir:
        raise ValueError(
            f"{cfg.name}: kd_teacher={cfg.kd_teacher!r} requires train(..., "
            f"teacher_dir=<dir containing that run's ckpt_best.pt>)"
        )
    ckpt = Path(teacher_dir) / "ckpt_best.pt"
    if not ckpt.exists():
        raise FileNotFoundError(
            f"{cfg.name}: no ckpt_best.pt in {teacher_dir} for teacher "
            f"{cfg.kd_teacher!r}"
        )
    state = torch.load(ckpt, map_location="cpu", weights_only=False)
    teacher = build_model("whisper")
    teacher.load_state_dict(state["model"] if "model" in state else state)
    teacher = teacher.to(device).eval().requires_grad_(False)
    print(f"teacher {cfg.kd_teacher}: {count_params(teacher):,} params "
          f"from {ckpt} (step {state.get('step')}, "
          f"val AUC {state.get('val_auc')})")
    return teacher


# ---------------- training ----------------

def train(cfg: ExperimentConfig, sources: dict, out_dir: str,
          device: str | None = None, steps_per_epoch: int | None = None,
          num_workers: int = 2, time_budget_minutes: float | None = None,
          teacher_dir: str | None = None):
    """sources: {"train": [(manifest, root), ...], "val": ..., "test": ...}

    time_budget_minutes: stop cleanly at the next checkpoint once this much
    wall-clock has elapsed, returning {"status": "time_budget_reached", ...}
    without final eval/export. Kaggle publishes no output from a commit run
    that hits the 12 h wall, so a run that would overrun must end itself.
    Not part of cfg.config_hash() — resuming with a different budget is fine.

    teacher_dir: directory holding cfg.kd_teacher's ckpt_best.pt. Mandatory
    when cfg.kd_teacher is set, ignored otherwise.
    """
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)

    train_df = load_manifests(sources["train"], "train")
    val_df = load_manifests(sources["val"], "val")
    print(f"train rows: {train_df.height}, val rows: {val_df.height}")
    train_ds = TurnDataset(train_df, cfg, train=True)
    val_ds = TurnDataset(val_df, cfg, train=False)

    spe = steps_per_epoch or math.ceil(train_df.height / cfg.batch_size)
    total_steps = spe * cfg.epochs
    warmup = max(1, int(total_steps * cfg.warmup_frac))

    model = build_model(cfg.arch).to(device)
    print(f"arch={cfg.arch} params={count_params(model):,}")
    mel_fn = LogMel().to(device)

    enc_params = [p for n, p in model.named_parameters() if n.startswith("encoder.")]
    other_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
    opt = torch.optim.AdamW(
        [{"params": enc_params, "lr": cfg.lr_encoder},
         {"params": other_params, "lr": cfg.lr_head}],
        weight_decay=cfg.weight_decay,
    )

    def lr_lambda(step):
        if step < warmup:
            return step / warmup
        p = (step - warmup) / max(1, total_steps - warmup)
        return 0.5 * (1 + math.cos(math.pi * p))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = torch.amp.GradScaler(enabled=(device == "cuda"))
    loss_fn = torch.nn.BCEWithLogitsLoss()

    # teacher is loaded before the resume block so a bad teacher path fails in
    # seconds rather than after the first epoch
    teacher = load_teacher(cfg, teacher_dir, device) if cfg.kd_teacher else None

    def batch_loss(mel, label):
        """Hard-label BCE, blended with the teacher's soft targets for KD.

        The teacher sees the identical (augmented) mel batch the student does —
        standard for response-based distillation, and it means the soft target
        already accounts for the pause/noise/speed augmentation applied here.
        """
        logits = model(mel)
        hard = loss_fn(logits, label)
        if teacher is None:
            return logits, hard
        t = cfg.kd_temperature
        with torch.no_grad():
            soft_target = torch.sigmoid(teacher(mel) / t)
        soft = loss_fn(logits / t, soft_target)
        return logits, cfg.kd_alpha * hard + (1 - cfg.kd_alpha) * soft

    # resume
    step, best_val_auc = 0, 0.0
    ckpt_last = out / "ckpt_last.pt"
    if ckpt_last.exists():
        state = torch.load(ckpt_last, map_location=device, weights_only=False)
        if state["cfg_hash"] == cfg.config_hash():
            model.load_state_dict(state["model"])
            opt.load_state_dict(state["opt"])
            sched.load_state_dict(state["sched"])
            scaler.load_state_dict(state["scaler"])
            step = state["step"]
            best_val_auc = state["best_val_auc"]
            # a kill between the ckpt_best and ckpt_last writes leaves
            # ckpt_last's best_val_auc stale; trust whichever is higher
            ckpt_best = out / "ckpt_best.pt"
            if ckpt_best.exists():
                b = torch.load(ckpt_best, map_location="cpu", weights_only=False)
                best_val_auc = max(best_val_auc, float(b.get("val_auc", 0.0)))
            print(f"resumed from step {step} (best val AUC {best_val_auc:.4f})")
        else:
            print("checkpoint config hash mismatch — starting fresh")

    history = []
    t_start = time.time()

    def budget_reached() -> bool:
        return (time_budget_minutes is not None
                and (time.time() - t_start) / 60.0 >= time_budget_minutes)

    def budget_stop(step: int) -> dict:
        print(f"TIME BUDGET REACHED at step {step}/{total_steps} — "
              f"outputs saved; resume next run", flush=True)
        return {"experiment": cfg.name, "status": "time_budget_reached",
                "step": step, "total_steps": total_steps}

    epoch_pass = step // spe
    while step < total_steps:
        train_ds.set_epoch(epoch_pass)
        loader = DataLoader(
            train_ds, batch_size=cfg.batch_size,
            sampler=train_ds.balanced_sampler(num_samples=spe * cfg.batch_size),
            num_workers=num_workers, pin_memory=(device == "cuda"),
            drop_last=True, persistent_workers=False,
        )
        model.train()
        for wav, label, _ in loader:
            if step >= total_steps:
                break
            mel = mel_fn(wav.to(device, non_blocking=True))
            label = label.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                _, loss = batch_loss(mel, label)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1

            if step % 50 == 0:
                print(f"step {step}/{total_steps} loss {loss.item():.4f} "
                      f"lr {sched.get_last_lr()[-1]:.2e} "
                      f"({(time.time() - t_start) / 60:.1f} min)", flush=True)
            if step % cfg.checkpoint_every_steps == 0:
                save_ckpt(ckpt_last, model, opt, sched, scaler, step,
                          best_val_auc, cfg.config_hash())
                if budget_reached():
                    return budget_stop(step)

            if step % spe == 0:  # epoch boundary -> validate
                vl, vp = predict(model, mel_fn, val_ds, device,
                                 cfg.batch_size, num_workers)
                val_auc = rank_auc(vl, vp)
                val_acc = float(((vp >= 0.5) == vl).mean())
                history.append({"step": step, "val_auc": round(val_auc, 4),
                                "val_acc_050": round(val_acc, 4)})
                print(f"  == step {step}: val AUC {val_auc:.4f} "
                      f"acc@0.5 {val_acc:.4f}", flush=True)
                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    atomic_save({"model": model.state_dict(),
                                 "cfg_hash": cfg.config_hash(),
                                 "step": step, "val_auc": val_auc},
                                out / "ckpt_best.pt")
                save_ckpt(ckpt_last, model, opt, sched, scaler, step,
                          best_val_auc, cfg.config_hash())
                if budget_reached():
                    return budget_stop(step)
                model.train()
        epoch_pass += 1

    # ---- final evaluation with best weights ----
    best = torch.load(out / "ckpt_best.pt", map_location=device, weights_only=False)
    model.load_state_dict(best["model"])

    vl, vp = predict(model, mel_fn, val_ds, device, cfg.batch_size, num_workers)
    threshold = tune_threshold(vl, vp)

    test_df = load_manifests(sources["test"], "test")
    test_ds = TurnDataset(test_df, cfg, train=False)
    tl, tp = predict(model, mel_fn, test_ds, device, cfg.batch_size, num_workers)
    test_metrics = slice_metrics(test_df, tl, tp, threshold)

    metrics = {
        "experiment": cfg.name,
        "config": cfg.__dict__,
        "params": count_params(model),
        "train_rows": train_df.height,
        "best_val_auc": round(best_val_auc, 4),
        "history": history,
        "threshold": threshold,
        "test": test_metrics,
        "train_minutes": round((time.time() - t_start) / 60, 1),
    }
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print(json.dumps(test_metrics, indent=2))

    export_onnx(model, mel_fn, out, cfg, test_df, test_ds, threshold, metrics)
    return metrics


# ---------------- export ----------------

def export_onnx(model, mel_fn, out: Path, cfg, test_df, test_ds,
                threshold, metrics):
    import onnxruntime as ort
    model = model.cpu().eval()
    dummy = torch.randn(1, 80, 800)
    fp32_path = out / "model_fp32.onnx"
    # static batch=1: turn detection inference is streaming, one window at a
    # time, and dynamic batch breaks the bidirectional-GRU reshape on export
    try:  # legacy exporter: consistent shape metadata, quantizer-friendly
        torch.onnx.export(
            model, (dummy,), str(fp32_path),
            input_names=["mel"], output_names=["logit"],
            opset_version=17, dynamo=False,
        )
    except TypeError:  # older torch without the dynamo kwarg
        torch.onnx.export(
            model, (dummy,), str(fp32_path),
            input_names=["mel"], output_names=["logit"], opset_version=17,
        )

    # torch vs onnx parity (per-sample, batch=1 graph)
    sess = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
    diffs = []
    for _ in range(4):
        x = torch.randn(1, 80, 800)
        with torch.no_grad():
            ref = torch.sigmoid(model(x)).numpy()
        got = 1 / (1 + np.exp(-sess.run(None, {"mel": x.numpy()})[0]))
        diffs.append(np.abs(ref - got).max())
    parity = float(max(diffs))
    print(f"onnx fp32 parity: max |dprob| = {parity:.2e}")

    from onnxruntime.quantization import QuantType, quantize_dynamic
    int8_path = out / "model_int8.onnx"
    quantize_dynamic(str(fp32_path), str(int8_path), weight_type=QuantType.QUInt8)

    # int8 accuracy on a stratified test subset (bounded runtime on CPU)
    rng = np.random.default_rng(0)
    labels_np = test_df["label"].to_numpy()
    idx = np.concatenate([
        rng.permutation(np.nonzero(labels_np == c)[0])[:1000] for c in (0, 1)
    ])
    sess8 = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
    probs, labels = [], []
    mel_cpu = mel_fn.cpu()
    for i in idx:
        wav, label, _ = test_ds[int(i)]
        mel = mel_cpu(wav.unsqueeze(0)).numpy()
        logit = sess8.run(None, {"mel": mel})[0][0]
        probs.append(1 / (1 + np.exp(-logit)))
        labels.append(int(label))
    labels, probs = np.array(labels), np.array(probs).ravel()
    int8_metrics = {
        "n": len(labels),
        "acc_tuned": round(float(((probs >= threshold) == labels).mean()), 4),
        "auc": round(rank_auc(labels, probs), 4),
        "size_mb": round(int8_path.stat().st_size / 1e6, 2),
        "fp32_size_mb": round(fp32_path.stat().st_size / 1e6, 2),
        "fp32_parity_max_dprob": parity,
    }
    metrics["int8_subset"] = int8_metrics
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print("int8:", json.dumps(int8_metrics, indent=2))

In [ ]:
EXPERIMENT = "e1_baseline"   # e1_baseline | e2_hinglish_aug | e3_tinymel_scratch | e4_no_pause_aug | e5_distill | e6_full_data
PREP = "/kaggle/input/smart-turn-enhi-prep/prep"
HINGLISH = "/kaggle/input/hinglish-synth"
RESUME_FROM = ""             # e.g. "/kaggle/input/turn-detect-ckpt/run_e2_hinglish_aug"
TEACHER_FROM = ""            # e5_distill only: run folder holding the teacher's ckpt_best.pt
TIME_BUDGET_MIN = 630        # stop cleanly at 10.5 h; Kaggle kills at 12 h with NO output

In [ ]:
import shutil, sys
from pathlib import Path

import torch

sys.path.insert(0, ".")
from turn_detector.config import EXPERIMENTS
from turn_detector.train import train

cfg = EXPERIMENTS[EXPERIMENT]
out_dir = Path("/kaggle/working") / f"run_{EXPERIMENT}"
out_dir.mkdir(parents=True, exist_ok=True)

# fail in seconds, not sessions: Kaggle's torch has no sm_60 (P100) kernels
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    print(f"GPU: {name} (sm_{cap[0]}{cap[1]})")
    if cap[0] < 7:
        raise RuntimeError(
            f"{name} (sm_{cap[0]}{cap[1]}) is unsupported by this torch build — "
            f"session must use the T4 (machine_shape NvidiaTeslaT4). Re-push."
        )

if RESUME_FROM:
    # fail loudly: a silent fall-through here burns a whole GPU session
    # restarting from step 0 while the log still says "resuming"
    src_ckpt = Path(RESUME_FROM, "ckpt_last.pt")
    if not src_ckpt.exists():
        raise RuntimeError(
            f"RESUME_FROM={RESUME_FROM!r} has no ckpt_last.pt. Attach the right "
            f"input dataset/notebook output, or set RESUME_FROM = \"\" to start fresh."
        )
    for f in Path(RESUME_FROM).glob("*"):
        if not (out_dir / f.name).exists():
            shutil.copy(f, out_dir / f.name)
    head = torch.load(out_dir / "ckpt_last.pt", map_location="cpu", weights_only=False)
    if head["cfg_hash"] != cfg.config_hash():
        raise RuntimeError(
            f"checkpoint cfg_hash {head['cfg_hash']} != {cfg.config_hash()} for "
            f"{EXPERIMENT}: that checkpoint belongs to a different config. "
            f"Set RESUME_FROM = \"\" to start fresh."
        )
    print(f"resuming {EXPERIMENT} from {RESUME_FROM} @ step {head['step']}")

# Kaggle has used both flat (/kaggle/input/<slug>) and nested
# (/kaggle/input/{datasets,notebooks}/<user>/<slug>) mount layouts, and a
# source kernel's output takes minutes to publish after it completes — so
# resolve mounts by searching rather than trusting a hardcoded path.
import os

def resolve_mount(configured: str, slug: str, marker: str = "manifest.parquet") -> str:
    if Path(configured, marker).exists():
        return configured
    hits = []
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in ("audio", "__pycache__")]
        if root.count("/") > 8:
            dirs[:] = []
        if marker in files and slug in root:
            hits.append(root)
    if len(hits) == 1:
        print(f"mount for {slug}: {configured} -> {hits[0]}")
        return hits[0]
    listing = "\n".join(sorted(
        str(Path(r) / f) for r, _, fs in os.walk("/kaggle/input")
        for f in fs if r.count("/") <= 6
    )[:60])
    raise RuntimeError(
        f"could not resolve mount for {slug!r} (marker {marker}, hits={hits}).\n"
        f"/kaggle/input contains:\n{listing}\n"
        f"If the source kernel just finished, wait a few minutes and re-push."
    )

PREP = resolve_mount(PREP, "turn-detect-01-data-prep")
HINGLISH = resolve_mount(HINGLISH, "hinglish-synth")
if RESUME_FROM:
    RESUME_FROM = resolve_mount(RESUME_FROM, "run_" + EXPERIMENT,
                                marker="ckpt_last.pt")

# distillation: the teacher's ckpt_best.pt rides in on the turn-detect-ckpt
# dataset. Fail here rather than train a "distilled" student with no teacher.
teacher_dir = None
if getattr(cfg, "kd_teacher", ""):
    if not TEACHER_FROM:
        raise RuntimeError(
            f"{EXPERIMENT} distills from {cfg.kd_teacher!r} but TEACHER_FROM is "
            f"empty. Attach the turn-detect-ckpt dataset and set TEACHER_FROM = "
            f'"/kaggle/input/turn-detect-ckpt/run_{cfg.kd_teacher}", or push with '
            f"python -m tools.push_kaggle train {EXPERIMENT} --teacher {cfg.kd_teacher}"
        )
    teacher_dir = resolve_mount(TEACHER_FROM, "run_" + cfg.kd_teacher,
                                marker="ckpt_best.pt")
    print(f"teacher ({cfg.kd_teacher}): {teacher_dir}")

real = [(f"{PREP}/manifest.parquet", PREP)]
synth = [(f"{HINGLISH}/manifest.parquet", HINGLISH)]
train_sources = real + (synth if cfg.use_hinglish_synth else [])
sources = {
    "train": train_sources,
    "val": train_sources,
    "test": real + synth,   # always evaluate hinglish slice, even for e1
}

metrics = train(cfg, sources, str(out_dir), num_workers=3,
                time_budget_minutes=TIME_BUDGET_MIN, teacher_dir=teacher_dir)

if metrics.get("status") == "time_budget_reached":
    print(
        f"\nPARTIAL RUN: stopped at step {metrics['step']}/{metrics['total_steps']}.\n"
        f"ckpt_last.pt is in run_{EXPERIMENT}/ and this version WILL publish its output.\n"
        f"To continue:  python -m tools.push_kaggle train {EXPERIMENT} --resume\n"
        f"(or manually: attach this version's output, set\n"
        f" RESUME_FROM = \"/kaggle/input/turn-detect-ckpt/run_{EXPERIMENT}\", Save & Run All)\n"
        f"No metrics.json/ONNX yet — those are written by the final run."
    )

In [ ]:
import json
from pathlib import Path

run_dir = Path("/kaggle/working") / f"run_{EXPERIMENT}"
mpath = run_dir / "metrics.json"
if not mpath.exists():
    print(f"no metrics.json in {run_dir} — partial run (see the cell above). "
          f"Save Version so ckpt_last.pt is published, then resume.")
    print("files:", sorted(p.name for p in run_dir.glob("*")))
else:
    m = json.loads(mpath.read_text())
    head = {k: m[k] for k in ("experiment", "params", "best_val_auc",
                              "threshold", "train_minutes") if k in m}
    print(json.dumps(head, indent=2))
    print(json.dumps(m.get("test", {}), indent=2))
    print(json.dumps(m.get("int8_subset", {}), indent=2))
    print("\nNow: Save Version, then download run_" + EXPERIMENT + "/ into the repo's experiments/ folder.")